# v8 — LLM hakem: gri bölgeyi büyük LLM ile yeniden etiketle

**Tanı:** v1→v6 tüm modeller aynı etiket soyundan öğreniyor; model sınıfı atlaması +0.004, pseudo döngüsü −0.003, ensemble +0.000 verdi → tavan **etiket**, model değil. Gri bölgede (~%7, ~250K çift) tüm modeller aynı hatayı yapıyor.

**Yöntem:** Qwen2.5-32B (AWQ, vLLM ile yerel — API/ücret yok) gri bölge çiftlerini doğrudan yargılar; kararlar %25 sabit oranla birleştirilir.

## Gerekenler
1. v6 bitmiş olmalı: Drive'da `WORK/artifacts/ce_v6_test.npy` duruyor olmalı.
2. **A100** runtime (Runtime → Change runtime type). L4'te 14B modele düş (1. hücrede `LLM` değişkeni).
3. `WORK` yolunu v6'da kullandığınla AYNI yap (1. hücre).

## Süre (A100)
vLLM kurulum ~5 dk, model indirme ~5 dk, yargılama ~1.5–3 saat (250K çift, **resumable** — oturum koparsa hücreleri baştan çalıştır, kaldığı chunk'tan sürer).

## Akış
Hücreleri sırayla çalıştır. Önce `--limit 120` göz kontrolü: LLM'in örnek kararlarını oku, saçmalıyorsa devam etme.

In [ ]:
# 0) Parametreler — WORK'u v6'da kullandiginla AYNI yap!
COMPETITION = "trendyol-e-ticaret-yarismasi-2026-kaggle"
REPO_URL = "https://github.com/emrehasilik/trendyol_kaggle.git"
GITHUB_TOKEN = ""
WORK = "/content/drive/MyDrive/trendyol_kaggle"
LLM = "Qwen/Qwen2.5-32B-Instruct-AWQ"  # L4/hiz icin: "Qwen/Qwen2.5-14B-Instruct-AWQ"

In [ ]:
# 1) GPU kontrol
!nvidia-smi -L

In [ ]:
# 2) Drive + v6 skorlarinin varligini dogrula
from google.colab import drive
drive.mount("/content/drive")
import os
p = f"{WORK}/artifacts/ce_v6_test.npy"
assert os.path.exists(p), f"EKSIK: {p} — WORK yolunu v6'daki ile ayni yap (0. hucre)"
print("OK:", p)

In [ ]:
# 3) Repo
import os
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
else:
    !cd /content/repo && git pull -q
print("repo hazir")

In [ ]:
# 4) Ortam degiskenleri
import os
os.environ["TK2_WORK"] = WORK
os.environ["TK2_LLM"] = LLM
os.environ["TK2_HF_CACHE"] = "/content/hf_cache"
print("WORK =", WORK, "| LLM =", LLM)

In [ ]:
# 5) Yarisma verisi (Kaggle API)
import os
DATA = "/content/repo/data"
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/items.csv"):
    os.makedirs("/root/.kaggle", exist_ok=True)
    !cp {WORK}/kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c {COMPETITION} -p {DATA}
    !cd {DATA} && unzip -o -q "*.zip" && rm -f *.zip
!ls -lh {DATA}

In [ ]:
# 6) vLLM kurulumu (~5-10 dk): surucunun CUDA surumune uygun vllm+torch ikilisi
#    zorla kurulur (libcudart.so uyusmazligi fix). Sonda 'vllm import OK' gormelisin.
%pip install -q uv
import subprocess, re
_o = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
_cuda = float(re.search(r"CUDA Version:\s*([\d.]+)", _o).group(1))
print("surucu CUDA:", _cuda)
if _cuda >= 13:
    !uv pip install --system -q --reinstall vllm --torch-backend=auto
    # cu13 torch'un eksik birakabildigi runtime kutubhaneleri (libnvrtc.so.13 vb.)
    !uv pip install --system -q nvidia-cuda-nvrtc-cu13 nvidia-nvjitlink-cu13
else:
    !uv pip install --system -q --reinstall vllm==0.11.0 --torch-backend=cu128
!python -c "import vllm; print('vllm import OK,', vllm.__version__)"

In [ ]:
# 7) GOZ KONTROLU: 120 ornek yargila, dosya yazmaz (~10 dk, model indirme dahil)
# Ciktida ce=CE skoru, llm=LLM karari. Ornekleri OKU: kararlar mantikliysa devam.
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v6 --limit 120

In [ ]:
# 8) TAM KOSU: gri bolgeyi yargila (resumable; kopursa bu hucreyi tekrar calistir)
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v6

In [ ]:
# 9) Birlestir + submission (GPU gerekmez, saniyeler surer)
# alpha = LLM'e guven agirligi (gri bolgede). Iki dosya uret:
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v6 --alpha 0.7
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v6 --alpha 1.0

In [ ]:
# 10) Gonder: ONCE alpha 0.7 dosyasini. Yukselirse alpha 1.0'i da dene.
print("dosyalar:")
!ls -lh {WORK}/output/sub_v6_llm*.csv
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v6_llm70_rate25.csv -m "v6 + LLM hakem alpha0.7 rate25"